In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
path_to_data = '/content/drive/MyDrive/CEMS'
print(os.listdir(path_to_data))


Mounted at /content/drive
['EMSN194', 'EMSR352', 'EMSR416', 'EMSR466', 'EMSR339', 'EMSR468', 'EMSR417']


In [2]:
!pip install torch torchvision
!pip install rasterio  # for handling .tif images
!pip install albumentations  # for data augmentation


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.3/22.3 MB 126.2 MB/s eta 0:00:00


In [3]:
from torch.cuda.amp import autocast, GradScaler
scaler = GradScaler()

/tmp/ipython-input-2736003858.py:2: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [4]:
import os
import numpy as np
import torch
import rasterio
from torch.utils.data import Dataset

class SenForFloodEventDataset(Dataset):
    def __init__(self, root_dir, modalities, patch_size=512,
                 random_crop=False, transforms=None):
        self.root_dir = root_dir
        self.modalities = modalities
        self.patch_size = patch_size
        self.random_crop = random_crop
        self.transforms = transforms

        self.index = []  # list of (event_id, sample_id)

        # Go through all events
        events = sorted([
            f for f in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, f))
        ])

        print("🔍 Filtering samples (<5% flood removed)...")

        for event in events:
            before_path = os.path.join(root_dir, event, "s1_before_flood")
            tif_files = sorted([
                f for f in os.listdir(before_path)
                if f.endswith(".tif")
            ])

            # Extract sample prefixes
            sample_ids = [f.split("_")[0] for f in tif_files]

            for sid in sample_ids:

                # ----------------------------------------
                # LOAD MASK FIRST → FILTER BY FLOOD RATIO
                # ----------------------------------------
                mask_path = os.path.join(root_dir, event, "flood_mask")
                mask_file = [f for f in os.listdir(mask_path)
                             if f.startswith(sid)]

                if len(mask_file) != 1:
                    continue

                mask_file = os.path.join(mask_path, mask_file[0])
                with rasterio.open(mask_file) as src:
                    mask = src.read(1).astype("float32")

                flood_ratio = (mask > 0).mean()

                # Skip samples with < 5% water
                if flood_ratio < 0.05:
                    continue

                # Keep valid sample
                self.index.append((event, sid))

        print(f"✅ Samples kept after filtering: {len(self.index)}")

    def __len__(self):
        return len(self.index)

    def load_raster(self, event, folder, sample_id):
        folder_path = os.path.join(self.root_dir, event, folder)
        file = [f for f in os.listdir(folder_path)
                if f.startswith(sample_id)]
        if len(file) != 1:
            raise RuntimeError(f"Missing file for {event} {folder} {sample_id}")

        path = os.path.join(folder_path, file[0])
        with rasterio.open(path) as src:
            return src.read()

    def crop(self, img, mask):
        _, H, W = img.shape
        ph = self.patch_size

        if self.random_crop:
            top = np.random.randint(0, H - ph)
            left = np.random.randint(0, W - ph)
        else:
            top = (H - ph) // 2
            left = (W - ph) // 2

        return (img[:, top:top+ph, left:left+ph],
                mask[:, top:top+ph, left:left+ph])

    def __getitem__(self, idx):
        event, sid = self.index[idx]

        # Load all modalities
        arrays = []
        for folder in self.modalities:
            arrays.append(self.load_raster(event, folder, sid))
        x = np.concatenate(arrays, axis=0)

        # Load mask
        y = self.load_raster(event, "flood_mask", sid)
        y = (y > 0).astype(np.float32)

        # Crop
        x, y = self.crop(x, y)

        # Normalize each channel
        x = x.astype(np.float32)
        for c in range(x.shape[0]):
            m, s = np.nanmean(x[c]), np.nanstd(x[c]) + 1e-6
            x[c] = (x[c] - m) / s

        return torch.tensor(x), torch.tensor(y)

In [5]:
import rasterio

path = "/content/drive/MyDrive/CEMS/EMSN194/s1_before_flood/000000_s1_before_flood.tif"

with rasterio.open(path) as src:
    print("Shape (C, H, W):", src.count, src.height, src.width)
    print("CRS:", src.crs)
    print("Dtype:", src.dtypes)

Shape (C, H, W): 4 512 512
CRS: EPSG:3857
Dtype: ('float32', 'float32', 'float32', 'float32')


In [6]:
from torch.utils.data import DataLoader

# Instantiate dataset
ds = SenForFloodEventDataset(
    root_dir="/content/drive/MyDrive/CEMS",   # <-- update if needed
    modalities=[
        "s1_before_flood",
        "s1_during_flood",
        "s2_before_flood",
        "s2_during_flood",
        "terrain",
        "LULC",
    ],
    patch_size=256,
    random_crop=False    # for easy reproducibility
)

print("Total samples =", len(ds))

# Show first 5 (event, sample_id) pairs
print("\nIndex preview:")
for i in range(len(ds)):
    print(i, ds.index[i])

x, y = ds[0]
print("\nLoaded sample shapes:")
print("x:", x.shape)   # (C, H, W)
print("y:", y.shape)   # (1, H, W)

# Check stats
print("\nx stats: mean =", x.mean().item(), ", std =", x.std().item())
print("y unique values:", torch.unique(y))


🔍 Filtering samples (<5% flood removed)...
✅ Samples kept after filtering: 328
Total samples = 328

Index preview:
0 ('EMSN194', '000000')
1 ('EMSN194', '000001')
2 ('EMSN194', '000002')
3 ('EMSN194', '000003')
4 ('EMSN194', '000005')
5 ('EMSN194', '000006')
6 ('EMSN194', '000008')
7 ('EMSN194', '000009')
8 ('EMSN194', '000010')
9 ('EMSN194', '000011')
10 ('EMSN194', '000012')
11 ('EMSN194', '000013')
12 ('EMSN194', '000017')
13 ('EMSN194', '000018')
14 ('EMSN194', '000019')
15 ('EMSN194', '000020')
16 ('EMSN194', '000021')
17 ('EMSN194', '000023')
18 ('EMSN194', '000024')
19 ('EMSN194', '000026')
20 ('EMSN194', '000027')
21 ('EMSN194', '000028')
22 ('EMSR339', '000000')
23 ('EMSR339', '000001')
24 ('EMSR339', '000002')
25 ('EMSR339', '000004')
26 ('EMSR339', '000005')
27 ('EMSR339', '000006')
28 ('EMSR339', '000007')
29 ('EMSR339', '000008')
30 ('EMSR339', '000009')
31 ('EMSR339', '000010')
32 ('EMSR339', '000011')
33 ('EMSR352', '000000')
34 ('EMSR352', '000001')
35 ('EMSR352', '0000


Loaded sample shapes:
x: torch.Size([27, 256, 256])
y: torch.Size([1, 256, 256])

x stats: mean = -1.6522351486969455e-08 , std = 0.9229581952095032
y unique values: tensor([0., 1.])


In [7]:
from torch.utils.data import DataLoader

dataset = SenForFloodEventDataset(
    root_dir="/content/drive/MyDrive/CEMS",
    patch_size=256,
    random_crop=False,
    modalities=[
        "s1_before_flood",
        "s1_during_flood",
        "s2_before_flood",
        "s2_during_flood",
        "terrain",
        "LULC",
    ],
)

# Split dataset into training and validation
num_samples = len(dataset)
train_size = int(0.8 * num_samples)
val_size = num_samples - train_size

train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

print(f"Training samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}")

# Loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# Test one batch
for x_batch, y_batch in train_loader:
    print("x_batch shape:", x_batch.shape)
    print("y_batch shape:", y_batch.shape)
    break

🔍 Filtering samples (<5% flood removed)...
✅ Samples kept after filtering: 328
Training samples: 262, Validation samples: 66


x_batch shape: torch.Size([4, 27, 256, 256])
y_batch shape: torch.Size([4, 1, 256, 256])


In [8]:
print(len(train_loader.dataset))


262


In [9]:
!pip install segmentation-models-pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 6.1 MB/s eta 0:00:00


In [10]:
import os
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm
from sklearn.metrics import jaccard_score
import numpy as np
import segmentation_models_pytorch as smp

In [11]:
# ----------------------
# Model building blocks
# ----------------------
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.double_conv(x)


In [12]:
class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.pool_conv = nn.Sequential(nn.MaxPool2d(2), DoubleConv(in_ch, out_ch))
    def forward(self, x):
        return self.pool_conv(x)

In [13]:
class Up(nn.Module):
    def __init__(self, in_ch, out_ch, bilinear=True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_ch, out_ch)
        else:
            # convtranspose approach (less used here)
            self.up = nn.ConvTranspose2d(in_ch//2, in_ch//2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_ch, out_ch)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        # pad if needed
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                        diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

In [14]:
class OutConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size=1)
    def forward(self, x):
        return self.conv(x)

In [15]:
class UNetMultiModal(nn.Module):
    def __init__(self, in_channels=8, n_classes=1, bilinear=True):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = OutConv(64, n_classes if n_classes>1 else 1)

    def forward(self, x):
        x1 = self.inc(x)        # (B,64,H,W)
        x2 = self.down1(x1)     # (B,128,H/2,W/2)
        x3 = self.down2(x2)     # (B,256,H/4,W/4)
        x4 = self.down3(x3)     # (B,512,H/8,W/8)
        x5 = self.down4(x4)     # (B,512 or 1024,H/16,W/16)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        logits = self.outc(x)
        return logits

In [16]:
# ----------------------
# Losses: Dice + Focal
# ----------------------
def dice_loss(pred, target, eps=1e-6):
    # pred: logits or probabilities. We will apply sigmoid inside.
    if pred.ndim == 4 and pred.shape[1] > 1:
        # multi-class improbable here; take class 1
        pred_p = torch.softmax(pred, dim=1)[:,1:2]
    else:
        pred_p = torch.sigmoid(pred)
    if target.ndim == 3:
        target = target.unsqueeze(1).float()
    else:
        target = target.float()
    inter = (pred_p * target).sum(dim=(2,3))
    denom = pred_p.sum(dim=(2,3)) + target.sum(dim=(2,3))
    dice = (2. * inter + eps) / (denom + eps)
    return 1 - dice.mean()

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.bce = nn.BCEWithLogitsLoss(reduction='none')
        self.reduction = reduction
    def forward(self, logits, targets):
        if logits.ndim==4 and logits.shape[1]>1:
            logits = logits[:,1:2]
        if targets.ndim==3:
            targets = targets.unsqueeze(1).float()
        bce_loss = self.bce(logits, targets)
        p = torch.sigmoid(logits)
        pt = torch.where(targets == 1, p, 1 - p)
        w = (1 - pt) ** self.gamma
        loss = self.alpha * w * bce_loss
        return loss.mean() if self.reduction=='mean' else loss.sum()

class ComboLoss(nn.Module):
    def __init__(self, alpha=0.5):
        super().__init__()
        self.alpha = alpha
        self.focal = FocalLoss(alpha=0.25, gamma=2.0)
    def forward(self, logits, targets):
        d = dice_loss(logits, targets)
        f = self.focal(logits, targets)
        return self.alpha * f + (1 - self.alpha) * d

In [17]:
# ----------------------
# Utilities: IoU per batch
# ----------------------
def batch_iou(pred_logits, target, threshold=0.5):
    # returns jaccard / IoU for binary
    if pred_logits.ndim==4 and pred_logits.shape[1]>1:
        pred = torch.argmax(pred_logits, dim=1)
    else:
        pred = (torch.sigmoid(pred_logits) > threshold).long().squeeze(1)
    if target.ndim==4:
        target = target.squeeze(1)
    pred_np = pred.detach().cpu().numpy().ravel()
    targ_np = target.detach().cpu().numpy().ravel()
    # handle degenerate case
    try:
        return jaccard_score(targ_np, pred_np, average='binary', zero_division=1)
    except Exception:
        return 0.0

In [18]:
import torch.nn as nn
import segmentation_models_pytorch as smp

class Preprocessor(nn.Module):
    def __init__(self, in_ch=27, out_ch=8):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)

class FloodSegModel(nn.Module):
    def __init__(self, raw_in_channels=27, compressed_channels=8):
        super().__init__()

        # 27 → 8 learnable reduction
        self.pre = Preprocessor(
            in_ch=raw_in_channels,
            out_ch=compressed_channels
        )

        # DeepLabV3+ takes the reduced 8-channel tensor
        self.seg = smp.DeepLabV3Plus(
            encoder_name="mobilenet_v2",
            encoder_weight="imagenet",        # because 8-channels
            in_channels=compressed_channels,
            classes=1,
            activation=None
        )

    def forward(self, x):
        x = self.pre(x)
        x = self.seg(x)
        return x

In [19]:
# ----------------------
# Training config
# ----------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
in_channels = 27        # CHANGE if your channels differ
n_classes = 1
epochs = 25             # change as needed
learning_rate = 1e-4
weight_decay = 1e-5
checkpoint_dir = '/mnt/data/checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

model = FloodSegModel(raw_in_channels=27, compressed_channels=8).to(device)
criterion = ComboLoss(alpha=0.5)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)
scaler = GradScaler()

print(f"Device: {device} — Model params (M): {sum(p.numel() for p in model.parameters())/1e6:.2f}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/14.2M [00:00<?, ?B/s]

Device: cuda — Model params (M): 4.38


/tmp/ipython-input-1198418923.py:17: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [20]:
# ----------------------
# TRAIN / VAL loop
# ----------------------
best_val_iou = 0.0
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    t0 = time.time()
    pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Train E{epoch+1}/{epochs}")
    for i, batch in pbar:
        # Expect batch = (images, masks)
        images, masks = batch[0].to(device), batch[1].to(device)
        optimizer.zero_grad()
        with autocast():
            logits = model(images)
            loss = criterion(logits, masks)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()
        pbar.set_postfix({'loss': running_loss/(i+1)})
    train_time = time.time() - t0

    # Validation
    model.eval()
    val_loss = 0.0
    val_iou = 0.0
    with torch.no_grad():
        for j, batch in enumerate(val_loader):
            images, masks = batch[0].to(device), batch[1].to(device)
            with autocast():
                logits = model(images)
                loss = criterion(logits, masks)
            val_loss += loss.item()
            val_iou += batch_iou(logits, masks)
    val_loss = val_loss / max(1, len(val_loader))
    val_iou = val_iou / max(1, len(val_loader))
    scheduler.step(val_iou)
    print(f"Epoch {epoch+1}/{epochs}  train_loss {running_loss/len(train_loader):.4f}  val_loss {val_loss:.4f}  val_iou {val_iou:.4f}  time {train_time:.1f}s")

    # checkpoint best
    if val_iou > best_val_iou:
        best_val_iou = val_iou
        ckpt_path = os.path.join(checkpoint_dir, f'best_epoch{epoch+1}_iou{val_iou:.4f}.pth')
        torch.save({
            'epoch': epoch+1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scaler': scaler.state_dict(),
            'val_iou': val_iou
        }, ckpt_path)
        print(f"Saved best checkpoint: {ckpt_path}")

Train E1/10:   0%|          | 0/66 [00:00<?, ?it/s]WARNING:rasterio._env:CPLE_AppDefined in 000212_s2_before_flood.tif: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
/tmp/ipython-input-2064616961.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Train E1/10: 100%|██████████| 66/66 [03:44<00:00,  3.41s/it, loss=0.329]
/tmp/ipython-input-2064616961.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipython-input-2064616961.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipython-input-2064616961.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use

Epoch 1/10  train_loss 0.3289  val_loss 0.2830  val_iou 0.5174  time 225.0s
Saved best checkpoint: /mnt/data/checkpoints/best_epoch1_iou0.5174.pth


Train E2/10:   0%|          | 0/66 [00:00<?, ?it/s]WARNING:rasterio._env:CPLE_AppDefined in 000008_s2_before_flood.tif: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
/tmp/ipython-input-2064616961.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Train E2/10: 100%|██████████| 66/66 [01:05<00:00,  1.01it/s, loss=0.277]
/tmp/ipython-input-2064616961.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipython-input-2064616961.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipython-input-2064616961.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use

Epoch 2/10  train_loss 0.2772  val_loss 0.2560  val_iou 0.5491  time 65.4s
Saved best checkpoint: /mnt/data/checkpoints/best_epoch2_iou0.5491.pth


/tmp/ipython-input-2064616961.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Train E3/10:   0%|          | 0/66 [00:00<?, ?it/s]WARNING:rasterio._env:CPLE_AppDefined in 000107_s2_before_flood.tif: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
/tmp/ipython-input-2064616961.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Train E3/10: 100%|██████████| 66/66 [01:08<00:00,  1.04s/it, loss=0.251]
/tmp/ipython-input-2064616961.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipython-input-2064616961.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use

...

Train E9/10:   0%|          | 0/66 [00:00<?, ?it/s]WARNING:rasterio._env:CPLE_AppDefined in 000234_s2_before_flood.tif: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
/tmp/ipython-input-2064616961.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Train E9/10: 100%|██████████| 66/66 [01:19<00:00,  1.20s/it, loss=0.169]
/tmp/ipython-input-2064616961.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipython-input-2064616961.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipython-input-2064616961.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use

Epoch 9/10  train_loss 0.1694  val_loss 0.1563  val_iou 0.7213  time 79.3s
Saved best checkpoint: /mnt/data/checkpoints/best_epoch9_iou0.7213.pth


/tmp/ipython-input-2064616961.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Train E10/10:   0%|          | 0/66 [00:00<?, ?it/s]WARNING:rasterio._env:CPLE_AppDefined in 000188_s2_before_flood.tif: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
/tmp/ipython-input-2064616961.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Train E10/10: 100%|██████████| 66/66 [01:25<00:00,  1.29s/it, loss=0.167]
/tmp/ipython-input-2064616961.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipython-input-2064616961.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please u

Epoch 10/10  train_loss 0.1670  val_loss 0.1667  val_iou 0.7045  time 85.2s


/tmp/ipython-input-2064616961.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


In [25]:
# ----------------------
# Save final model
# ----------------------
final_path = os.path.join(checkpoint_dir, 'final_model2.pth')
torch.save({'model_state_dict': model.state_dict()}, final_path)
print("Training finished. Final model saved to:", final_path)

Training finished. Final model saved to: /mnt/data/checkpoints/final_model2.pth


In [22]:
# ----------------------
# Inference helper
# ----------------------
def predict_mask(model, image_tensor, device=device, threshold=0.5):
    # image_tensor: (C,H,W) or (1,C,H,W)
    model.eval()
    with torch.no_grad():
        if image_tensor.ndim == 3:
            x = image_tensor.unsqueeze(0).to(device)
        else:
            x = image_tensor.to(device)
        logits = model(x)
        probs = torch.sigmoid(logits)
        mask = (probs > threshold).long().squeeze(0).squeeze(0).cpu().numpy()
        probs_np = probs.squeeze(0).squeeze(0).cpu().numpy()
    return mask, probs_np

In [23]:
# Visualization (optional)
def show_prediction(sample_image_np, pred_mask, gt_mask=None):
    # sample_image_np: H,W,C (e.g., RGB preview) - if >3 channels, pass [:,:,:3]
    import matplotlib.pyplot as plt
    fig_count = 3 if gt_mask is not None else 2
    fig, axes = plt.subplots(1, fig_count, figsize=(12, 4))
    if sample_image_np.shape[2] >= 3:
        axes[0].imshow(sample_image_np[...,:3])
    else:
        axes[0].imshow(sample_image_np[...,0], cmap='gray')
    axes[0].set_title('Input (preview)'); axes[0].axis('off')
    axes[1].imshow(pred_mask, cmap='gray'); axes[1].set_title('Predicted mask'); axes[1].axis('off')
    if gt_mask is not None:
        axes[2].imshow(gt_mask, cmap='gray'); axes[2].set_title('Ground truth'); axes[2].axis('off')
    plt.tight_layout()
    plt.show()


In [26]:
!cp /mnt/data/checkpoints/final_model2.pth /content/drive/MyDrive/